In [ ]:
import numpy as np
import pandas as pd
import pod_models
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

plt.rcParams.update({
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 600,
    'figure.constrained_layout.use': True,
})

color_Campbell = "#b6321c" # BrickRed
color_DNVGL = "#f58137" # Orange
color_CV = "#000000"
color_prior = "#0071bc" # RoyalBlue

color_0_5 = "#C0C0C0" # light gray
color_1_5 = "#808080" # medium gray
color_3_0 = "#404040" # dark gray
color_17_0 = "#000000" # black

color_hit = "#00b903" # green
color_miss = "#ff001e" # red
color_fp = "#006dff" # blue

color_hit  = "#006d02"
color_miss = "#a60014"
color_fp = "#0043a0"

# color_hit  = "#009b55" # ForestGreen
# color_miss = "#b6321c" # BrickRed
# color_fp = "#0071bc" # RoyalBlue

# color_hit = "#008b72" # PineGreen
# color_miss = "#af3235" # Marroon
# color_fp = "#006eb8" # NavyBlue

text_width = 5.3 # Document text width in inches

In [ ]:
all_data = pd.read_excel("pod_data.xlsx")
N = len(all_data)
print(f"Total rows in dataset: {N}")

a = all_data["length_mm"]
r = all_data["length_pix"] / all_data["length_mm"]

a_hat = all_data["Detection_hit_miss"].to_numpy()

fp_probs = np.array([
    float(num) for num in np.concatenate([
        row.split(";") for row in
        all_data["fp_probs"].dropna().to_list()
    ])
])

# CV Method Detection Threshold Selection

## Figure 5

In [ ]:
thresholds = np.linspace(0.01, 1, 300)

fn_rates = np.sum(a_hat[:, None] <= thresholds[None, :], axis=0) / N
fp_rates = np.sum(fp_probs[:, None] > thresholds[None, :], axis=0) / N

fig, ax = plt.subplots(figsize=(0.7 * text_width, 0.5 * text_width))

points = np.column_stack([fp_rates, fn_rates])
segments = np.stack([points[:-1], points[1:]], axis=1)
norm = Normalize(vmin=thresholds.min(), vmax=thresholds.max())
lc = LineCollection(segments, cmap='viridis', norm=norm, linewidth=1.5)
lc.set_array(0.5 * (thresholds[:-1] + thresholds[1:]))
ax.add_collection(lc)

a_dec = 0.01
detected = a_hat > a_dec
fn_dec = np.sum(a_hat <= a_dec) / N
fp_dec = np.sum(fp_probs > a_dec) / N
ax.scatter(
    [fp_dec],
    [fn_dec],
    s=60,
    marker='D',
    c=[a_dec],
    cmap='viridis',
    norm=norm,
    edgecolors='black',
    linewidths=0.5,
    zorder=5,
    label=f'$\\hat{{a}}_\\mathrm{{th}} = {a_dec}$',
)

x_max = max(float(np.max(fp_rates)), float(fp_dec))
ax.set_xlim(-0.05, min(1.0, x_max * 1.05 if x_max > 0 else 1.0))
ax.set_ylim(0, 1.0)
ax.tick_params(axis='x', rotation=45)
ax.set_xticks(np.linspace(0, 1, 11))
ax.set_yticks(np.linspace(0, 1, 11))

# As the threshold increases, the curve moves left and up.
i_low = len(thresholds) // 5
i_high = 3 * len(thresholds) // 5
arrow_start = (fp_rates[i_low] + 0.1, max(0.0, fn_rates[i_low] - 0.015 + 0.1))
arrow_end = (fp_rates[i_high] + 0.1, min(1.0, fn_rates[i_high] + 0.03 + 0.1))

cbar = fig.colorbar(lc, ax=ax)
cbar.set_label('Threshold')

ax.set_xlabel('False positive rate')
ax.set_ylabel('False negative rate')
ax.legend()
ax.grid(True, alpha=0.3)
fig.savefig("Figures/threshold_selection.png")


## Figure 4

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(text_width, 0.55 * text_width))

marker_alpha = 0.85
marker_size = 6

ax.scatter(
    a[detected], a_hat[detected],
    c=color_hit, s=marker_size, alpha=marker_alpha, linewidths=0, label='True positive'
)

ax.scatter(
    a[~detected], a_hat[~detected],
    c=color_miss, s=marker_size, alpha=marker_alpha, linewidths=0, label='False negative'
)

fp_x_zero = np.zeros_like(fp_probs)
ax.scatter(
    fp_x_zero, fp_probs, 
    c=color_fp, s=marker_size, alpha=marker_alpha, linewidths=0, label='False positive'
)

ax.axhline(y=a_dec, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='$\\hat{a}_\\mathrm{th}$')

ax.set_xlabel('Crack length (mm)')
ax.set_ylabel('Detection confidence')
ax.set_ylim(-0.1, 1.1)
ax.set_xticks(np.arange(0, 1501, 100))
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 0.21))

fig.savefig("Figures/detection_vs_crack_length.png")

# Data Distribution

## Figure 2

In [ ]:
def plot_data_histogram(x_data, y_data, x_hit, y_hit, x_miss, y_miss, x_label, y_label, title):

    fig = plt.figure(figsize=(0.49 * text_width, 0.49 * 0.8 * text_width), layout="tight")

    gs = fig.add_gridspec(
        2, 2,
        width_ratios=[4, 1],
        height_ratios=[1, 4],
        left=0.08,
        bottom=0.08,
        right=0.82,
        top=0.94,
        wspace=0.03,
        hspace=0.03,
    )


    ax_main = fig.add_subplot(gs[1, 0])

    ax_main.scatter(
        x_hit,
        y_hit,
        c=color_hit,
        alpha=0.5,
        s=10,
        edgecolors="black",
        linewidths=0.4,
        label="Hit",
    )

    ax_main.scatter(
        x_miss,
        y_miss,
        c=color_miss,
        alpha=0.5,
        s=10,
        edgecolors="black",
        linewidths=0.4,
        label="Miss",
    )

    ax_main.set_xlabel(x_label)
    ax_main.set_ylabel(y_label)
    ax_main.set_xticks(np.arange(0, 1501, 250))
    ax_main.tick_params(axis='x', rotation=45)
    ax_main.grid(alpha=0.35)

    ax_main.legend(
        loc="upper left",
        bbox_to_anchor=(0.97, 1.305),
        frameon=True,
    )


    ax_top = fig.add_subplot(gs[0, 0], sharex=ax_main)

    ax_top.hist(
        [x_hit, x_miss],
        bins=np.histogram_bin_edges(x_data, bins=30),
        stacked=True,
        alpha=0.5,
        color=[color_hit, color_miss],
        edgecolor="black",
        linewidth=0.4,
    )

    ax_top.tick_params(labelbottom=False, bottom=False)
    ax_top.set_ylabel("Count", rotation=0, labelpad=28)
    ax_top.grid(axis="y", alpha=0.35)
    ax_top.yaxis.set_major_locator(MaxNLocator(nbins=3))

    ax_top.spines["top"].set_visible(False)
    ax_top.spines["right"].set_visible(False)


    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

    ax_right.hist(
        [y_hit, y_miss],
        bins=np.histogram_bin_edges(y_data, bins=30),
        stacked=True,
        orientation="horizontal",
        alpha=0.5,
        color=[color_hit, color_miss],
        edgecolor="black",
        linewidth=0.4,
    )

    ax_right.tick_params(labelleft=False, left=False)
    ax_right.set_xlabel("Count")
    ax_right.grid(axis="x", alpha=0.30)
    ax_right.tick_params(axis='x', rotation=45)
    ax_right.xaxis.set_major_locator(MaxNLocator(nbins=3))

    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)
    ax_right.spines["bottom"].set_visible(False)

    fig.savefig(f"Figures/{title}.png", bbox_inches="tight")

In [ ]:
a_pix = all_data["length_pix"]

plot_data_histogram(
    a, a_pix,
    a[detected], a_pix[detected],
    a[~detected], a_pix[~detected],
    "Crack length (mm)", "Crack length (pix)", 
    "stats_crack_length_mm_vs_crack_length_pix"
)

In [ ]:
plot_data_histogram(
    a, r,
    a[detected], r[detected],
    a[~detected], r[~detected], 
    "Crack length (mm)", "Resolution (pix/mm)", 
    "stats_crack_length_mm_vs_resolution"
)

# PoD Transform Selection

## Figure 12

In [ ]:
# Load extended data for model including the effect of resolution
all_data = pd.read_excel("pod_data_extended.xlsx")
N = len(all_data)
print(f"Total rows in dataset: {N}")

a = all_data["length_mm"]
r = all_data["length_pix"] / all_data["length_mm"]

a_hat = all_data["Detection_hit_miss"].to_numpy()

detected = a_hat > a_dec

In [ ]:
def transformation_cross_validation(data, k):
    folds = np.array_split(data, k, axis=1)
    ll = np.zeros((k, 2, 2))
    for m in range(k):
        validation = folds[m]
        training = np.hstack(folds[:m] + folds[m+1:])

        A, R, training_Z = training
        t_A_id = A
        t_A_log = np.log(A)
        t_R_id = R
        t_R_log = np.log(R)
        A, R, validation_Z = validation
        v_A_id = A
        v_A_log = np.log(A)
        v_R_id = R
        v_R_log = np.log(R)

        for i, (t_a, v_a) in enumerate(zip([t_A_id, t_A_log], [v_A_id, v_A_log])):
            for j, (t_r, v_r) in enumerate(zip([t_R_id, t_R_log], [v_R_id, v_R_log])):
                training_X = np.stack([t_a, t_r], axis=-1)
                validation_X = np.stack([v_a, v_r], axis=-1)
                ll[m, i, j] = pod_models.train_and_validate(training_X, training_Z, validation_X, validation_Z)

    return ll.mean(axis=0)

n = 100
ll = np.zeros((n, 2, 2))
for i in tqdm(range(n)):
    rng = np.random.default_rng(i)
    data = np.stack([a, r, detected], axis=0)
    rng.shuffle(data, axis=1)
    ll[i] = transformation_cross_validation(data, 10)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(0.8 * text_width, 0.8 * 0.6 * text_width))
labels = [r"\operatorname{id}", r"\log"]
for i, lA in enumerate(labels):
    for j, lR in enumerate(labels):
        ax.hist(ll[..., i, j], label=f"$h_a = {lA}$, $h_r = {lR}$", alpha=0.8, density=True)
ax.set_ylabel("Density")
ax.set_xlabel("Log-likelihood")
ax.legend()
fig.savefig("Figures/selection_transforms_length_resolution.png");